# Sesión 7B · Mapa de calor dinámico: el COVID-19 distrito por distrito

> **¿Dónde golpeó el COVID-19 en el Perú, y cómo cambió entre 2020 y 2023?**

Un mapa de calor muestra dónde se concentra algo. Uno **dinámico** añade un
control de tiempo: se aprieta *play* y el mapa cambia de año en año.

Vamos a construir dos, con datos oficiales del Ministerio de Salud: uno de
**casos** y otro de **muertes**, a nivel de distrito.

| Paso | Qué hacemos |
|---|---|
| 1 | Descargar los datos abiertos del MINSA |
| 2 | Leer solo lo necesario y limpiar fechas y ubigeos |
| 3 | Pasar de personas a distritos: contar por distrito y año |
| 4 | Ponerle coordenadas a cada distrito |
| 5 | Convertir los conteos en intensidad de calor |
| 6 | Armar un cuadro por año y dibujar el mapa |

In [1]:
import os

import folium
import geopandas as gpd
import pandas as pd
import requests
from folium.plugins import HeatMapWithTime

## 1. Descargar los datos del MINSA

Son dos conjuntos de la [Plataforma Nacional de Datos Abiertos](https://www.datosabiertos.gob.pe),
con **una fila por persona**:

| Archivo | Qué trae | Tamaño |
|---|---|---|
| `positivos_covid.csv` | cada caso positivo, con la fecha del resultado | 312 MB |
| `fallecidos_covid.csv` | cada fallecido, con la fecha de fallecimiento | 20 MB |

Los dos traen el **ubigeo** del distrito, que es lo que permite llevarlos al mapa.

Se guardan en `data/.cache/`, una carpeta que git ignora: GitHub no acepta
archivos de más de 100 MB, y además se pueden volver a descargar cuando haga
falta.

In [2]:
CARPETA = "../../data/.cache/minsa"

ARCHIVOS = {
    "positivos_covid.csv": "https://files.minsa.gob.pe/s/qWfz3p5mJBtmMEp/download",
    "fallecidos_covid.csv": "https://files.minsa.gob.pe/s/t9AFqRbXw3F55Ho/download",
}

# El servidor del MINSA responde "403 Prohibido" a los programas que no se
# presentan como un navegador, así que se le indica uno
NAVEGADOR = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
             "(KHTML, like Gecko) Chrome/140.0 Safari/537.36")

os.makedirs(CARPETA, exist_ok=True)

La descarga se hace **una sola vez**: si el archivo ya está en la carpeta, se
salta. El archivo grande se escribe por pedazos de 1 MB, para no tener los
312 MB en memoria al mismo tiempo.

In [3]:
for nombre, url in ARCHIVOS.items():
    destino = os.path.join(CARPETA, nombre)

    if os.path.exists(destino):
        print(nombre, "ya estaba descargado")
        continue

    respuesta = requests.get(url, headers={"User-Agent": NAVEGADOR}, stream=True, timeout=120)
    respuesta.raise_for_status()      # si el servidor falló, se detiene aquí

    with open(destino, "wb") as archivo:
        for pedazo in respuesta.iter_content(chunk_size=1024 * 1024):
            archivo.write(pedazo)

    print(nombre, "descargado:", round(os.path.getsize(destino) / 1_000_000), "MB")

positivos_covid.csv ya estaba descargado
fallecidos_covid.csv ya estaba descargado


## 2. Leer solo lo necesario

El archivo de positivos tiene 4.5 millones de filas y 10 columnas. Para el
mapa bastan dos: **cuándo** y **dónde**. Con `usecols` pandas ni siquiera
carga las demás.

Y lo de siempre: todo como texto (`dtype=str`), para que el ubigeo `040101`
no pierda su cero.

In [4]:
positivos = pd.read_csv(
    os.path.join(CARPETA, "positivos_covid.csv"),
    sep=";",
    usecols=["FECHA_RESULTADO", "UBIGEO"],
    dtype=str,
)
print(positivos.shape)
positivos.head(3)

(4585360, 2)


,FECHA_RESULTADO,UBIGEO
0,20221207,240101
1,20230822,150113
2,20240108,220101


In [5]:
fallecidos = pd.read_csv(
    os.path.join(CARPETA, "fallecidos_covid.csv"),
    sep=";",
    usecols=["FECHA_FALLECIMIENTO", "UBIGEO"],
    dtype=str,
)
print(fallecidos.shape)
fallecidos.head(3)

(220918, 2)


,FECHA_FALLECIMIENTO,UBIGEO
0,20220219,240101
1,20210529,240101
2,20210623,230102


### El año sale de la fecha

La fecha viene como texto, `20221207`. El año son sus primeros cuatro
caracteres.

In [6]:
positivos["anio"] = positivos["FECHA_RESULTADO"].str[:4]
fallecidos["anio"] = fallecidos["FECHA_FALLECIMIENTO"].str[:4]

positivos["anio"].value_counts(dropna=False).sort_index()

anio
1899        394
2020    1022620
2021    1307875
2022    2132013
2023      93361
2024      27074
NaN        2023
Name: count, dtype: int64

Mirar antes de filtrar sirve: hay casos fechados en **1899** (un error de
digitación) y otros sin fecha. Ninguno entra al mapa. Nos quedamos con los
cuatro años que queremos comparar.

In [7]:
ANIOS = ["2020", "2021", "2022", "2023"]

positivos = positivos[positivos["anio"].isin(ANIOS)]
fallecidos = fallecidos[fallecidos["anio"].isin(ANIOS)]

print("Casos 2020-2023  :", len(positivos))
print("Muertes 2020-2023:", len(fallecidos))

Casos 2020-2023  : 4555869
Muertes 2020-2023: 220605


### Los casos sin distrito

No todos los registros dicen dónde ocurrieron. El MINSA los publica con el
distrito «EN INVESTIGACIÓN» y el ubigeo vacío. Sin ubigeo no hay forma de
ubicarlos en un mapa distrital, así que hay que saber **cuántos** se quedan
fuera antes de sacarlos.

In [8]:
casos_sin_distrito = positivos["UBIGEO"].isna().sum()
muertes_sin_distrito = fallecidos["UBIGEO"].isna().sum()

print("Casos sin distrito  :", casos_sin_distrito, "=", round(casos_sin_distrito / len(positivos) * 100, 1), "%")
print("Muertes sin distrito:", muertes_sin_distrito)

positivos = positivos.dropna(subset=["UBIGEO"])
fallecidos = fallecidos.dropna(subset=["UBIGEO"])

Casos sin distrito  : 234500 = 5.1 %
Muertes sin distrito: 5


Uno de cada veinte casos no tiene distrito (la mayoría, de Lima). El mapa
subestima un poco los casos; en las muertes el problema casi no existe.

## 3. De personas a distritos: contar

Cada fila es una persona. El mapa necesita un número por distrito y por año:
eso es un `groupby` y contar filas.

In [9]:
casos = positivos.groupby(["UBIGEO", "anio"]).size().reset_index(name="casos")
casos.head(3)

,UBIGEO,anio,casos
0,010101,2020,1979
1,010101,2021,4135
2,010101,2022,6465


In [10]:
muertes = fallecidos.groupby(["UBIGEO", "anio"]).size().reset_index(name="muertes")
muertes.head(3)

,UBIGEO,anio,muertes
0,010101,2020,48
1,010101,2021,150
2,010101,2022,22


Se juntan las dos tablas. Va `how="outer"` porque un distrito puede tener casos
en un año y ninguna muerte, o al revés: no se debe perder ninguno. Lo que
falta se llena con cero.

In [11]:
covid = casos.merge(muertes, on=["UBIGEO", "anio"], how="outer")
covid = covid.rename(columns={"UBIGEO": "ubigeo"})

covid["casos"] = covid["casos"].fillna(0).astype(int)
covid["muertes"] = covid["muertes"].fillna(0).astype(int)
covid["anio"] = covid["anio"].astype(int)

print(covid.shape)
covid.head()

(6795, 4)


,ubigeo,anio,casos,muertes
0,010101,2020,1979,48
1,010101,2021,4135,150
2,010101,2022,6465,22
3,010101,2023,133,3
4,010102,2020,4,0


### La historia en cuatro filas

In [12]:
covid.groupby("anio")[["casos", "muertes"]].sum()

,casos,muertes
anio,,
2020,973685,95326
2021,1237543,107962
2022,2022237,14742
2023,87904,2570


**2022 tuvo el doble de casos que 2020, y la sexta parte de las muertes.** Es
la ola de ómicron sobre una población ya vacunada. Por eso hacen falta los dos
mapas: casos y muertes no cuentan la misma historia.

La tabla resumida pesa unos pocos KB. Se guarda, y desde aquí ya no hacen
falta los 312 MB originales.

In [13]:
covid.to_csv("covid_distritos_anual.csv", index=False)
print("Guardado: covid_distritos_anual.csv")

Guardado: covid_distritos_anual.csv


## 4. Ponerle coordenadas a cada distrito

Un mapa de calor no pinta polígonos: pinta **puntos**. Hace falta un punto por
distrito.

`representative_point()` devuelve un punto que cae con seguridad dentro del
polígono. (El centro geométrico exacto puede caer fuera en distritos con forma
de herradura.)

In [14]:
distritos = gpd.read_file("../../data/07-mapas/distritos_peru.geojson")

punto = distritos.representative_point()
distritos["latitud"] = punto.y
distritos["longitud"] = punto.x

distritos[["ubigeo", "departamento", "distrito", "latitud", "longitud"]].head(3)

,ubigeo,departamento,distrito,latitud,longitud
0,060504,CAJAMARCA,GUZMANGO,-7.357948,-78.925503
1,061105,CAJAMARCA,EL PRADO,-7.044150,-79.001444
2,061109,CAJAMARCA,NIEPOS,-6.934420,-79.115658


### Verificar antes de cruzar

In [15]:
print("Mapa  :", distritos["ubigeo"].dtype, "| ejemplo:", distritos["ubigeo"].iloc[0])
print("Covid :", covid["ubigeo"].dtype, "| ejemplo:", covid["ubigeo"].iloc[0])
print("Coinciden:", len(set(distritos["ubigeo"]) & set(covid["ubigeo"])), "de", covid["ubigeo"].nunique())

Mapa  : object | ejemplo: 060504
Covid : object | ejemplo: 010101
Coinciden: 1872 de 1872


In [16]:
puntos = covid.merge(
    distritos[["ubigeo", "departamento", "distrito", "latitud", "longitud"]],
    on="ubigeo", how="left",
)

print("Casos antes y después del cruce  :", covid["casos"].sum(), puntos["casos"].sum())
print("Muertes antes y después del cruce:", covid["muertes"].sum(), puntos["muertes"].sum())
print("Filas sin coordenadas:", puntos["latitud"].isna().sum())

Casos antes y después del cruce  : 4321369 4321369
Muertes antes y después del cruce: 220600 220600
Filas sin coordenadas: 0


Los totales no cambiaron y ningún distrito quedó sin coordenadas: el cruce no
perdió nada.

## 5. La intensidad del calor

El mapa de calor pide, para cada punto, una **intensidad entre 0 y 1**. Lo
natural sería dividir cada conteo entre el máximo. El problema son los
extremos:

In [17]:
puntos.sort_values("casos", ascending=False)[["distrito", "departamento", "anio", "casos"]].head(5)

,distrito,departamento,anio,casos
4639,LIMA,LIMA,2022,108516
4687,JESUS MARIA,LIMA,2022,64567
4763,SAN JUAN DE LURIGANCHO,LIMA,2022,62318
4795,SANTIAGO DE SURCO,LIMA,2022,55936
4775,SAN MARTIN DE PORRES,LIMA,2022,48351


In [18]:
print("Máximo :", puntos["casos"].max())
print("Mediana:", puntos["casos"].median())

Máximo : 108516
Mediana: 35.0


El Cercado de Lima llegó a más de 100 mil casos en un año; el distrito típico,
a 35. Dividiendo entre el máximo, casi todo el país quedaría en 0.0003:
invisible. Solo se vería Lima.

La solución es sacar la **raíz cuadrada**, que acerca los extremos sin
borrarlos:

| Casos | Dividido entre el máximo | Con raíz cuadrada |
|---|---|---|
| 108,516 | 1.00 | 1.00 |
| 10,000 | 0.09 | 0.30 |
| 1,000 | 0.009 | 0.10 |

El máximo es **uno solo para los cuatro años**. Así un mismo tono significa lo
mismo en 2020 que en 2023, y los años se pueden comparar.

In [19]:
maximo_casos = puntos["casos"].max()
maximo_muertes = puntos["muertes"].max()

puntos["peso_casos"] = (puntos["casos"] / maximo_casos) ** 0.5
puntos["peso_muertes"] = (puntos["muertes"] / maximo_muertes) ** 0.5

puntos[["distrito", "anio", "casos", "peso_casos", "muertes", "peso_muertes"]].head()

,distrito,anio,casos,peso_casos,muertes,peso_muertes
0,CHACHAPOYAS,2020,1979,0.135044,48,0.104163
1,CHACHAPOYAS,2021,4135,0.195205,150,0.184136
2,CHACHAPOYAS,2022,6465,0.244083,22,0.070519
3,CHACHAPOYAS,2023,133,0.035009,3,0.026041
4,ASUNCION,2020,4,0.006071,0,0.000000


## 6. Un cuadro por año

`HeatMapWithTime` recibe una **lista de cuadros**, como una película. Cada
cuadro es la lista de puntos de un año, y cada punto es
`[latitud, longitud, intensidad]`.

```
[
  [[lat, lon, peso], [lat, lon, peso], ...],    ← 2020
  [[lat, lon, peso], [lat, lon, peso], ...],    ← 2021
  ...
]
```

In [20]:
cuadros_casos = []

for anio in [2020, 2021, 2022, 2023]:
    del_anio = puntos[(puntos["anio"] == anio) & (puntos["casos"] > 0)]
    cuadros_casos.append(del_anio[["latitud", "longitud", "peso_casos"]].values.tolist())
    print(anio, "→", len(del_anio), "distritos con casos")

2020 → 1799 distritos con casos
2021 → 1834 distritos con casos
2022 → 1818 distritos con casos
2023 → 1219 distritos con casos


In [21]:
cuadros_casos[0][:2]      # los dos primeros puntos del primer año

[[-6.24147997357187, -77.8506730310419, 0.13504422245237654],
 [-5.992515598715869, -77.75371387856987, 0.0060713197687775255]]

## 7. El mapa dinámico de casos

Tres decisiones antes de dibujar:

- **Fondo gris.** Sobre un mapa lleno de colores el calor no se distingue.
  `Esri.WorldGrayCanvas` es gris y no pide clave.
- **Un solo color, de claro a oscuro.** El degradado que trae por defecto es
  un arcoíris (azul, verde, amarillo, rojo), que no dice qué es más y qué es
  menos. Con un solo tono, más oscuro siempre es más.
- **Una leyenda.** Folium no le pone leyenda al mapa de calor, así que se
  añade una cajita de HTML.

In [22]:
MAPA_BASE = "Esri.WorldGrayCanvas"

AZULES = {0.2: "#9ec5f4", 0.5: "#3987e5", 0.8: "#184f95", 1.0: "#0d366b"}

# Una cajita de HTML que se dibuja encima del mapa. Los {huecos} se llenan después
LEYENDA = '''
<div style="position: fixed; top: 12px; right: 12px; z-index: 1000; width: 210px;
            background: white; padding: 10px 12px; border-radius: 6px;
            font: 13px sans-serif; color: #333; box-shadow: 0 1px 5px rgba(0,0,0,0.3);">
  <b>{titulo}</b>
  <div style="height: 10px; margin: 8px 0 3px; background: linear-gradient(to right, {colores});"></div>
  <span>menos</span><span style="float: right;">más</span>
  <div style="margin-top: 6px; font-size: 11px; color: #666;">
    Por distrito y año. Los distritos vecinos suman su calor.</div>
</div>
'''

In [ ]:
mapa_casos = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles=MAPA_BASE)

HeatMapWithTime(
    cuadros_casos,
    index=["2020", "2021", "2022", "2023"],      # lo que muestra el control en cada cuadro
    gradient=AZULES,
    radius=18,
    max_opacity=0.85,
).add_to(mapa_casos)

leyenda_casos = LEYENDA.format(titulo="Casos de COVID-19", colores=", ".join(AZULES.values()))
mapa_casos.get_root().html.add_child(folium.Element(leyenda_casos))

mapa_casos

El control de abajo a la izquierda es el reproductor: ▶ recorre los años, y
la barra permite saltar a uno. Al acercarse a una ciudad, la mancha se separa
en sus distritos.

In [24]:
mapa_casos.save("mapa_covid_casos.html")
print("Guardado: mapa_covid_casos.html")

Guardado: mapa_covid_casos.html


## 8. Lo mismo para las muertes

Va en un **mapa aparte**. Poner dos `HeatMapWithTime` en el mismo mapa no
funciona: cada uno crea su propio reproductor, y los dos quedan
desincronizados (uno avanza a 2023 mientras el otro sigue en 2020).

Cambian tres cosas: la columna, el color y el título.

In [25]:
cuadros_muertes = []

for anio in [2020, 2021, 2022, 2023]:
    del_anio = puntos[(puntos["anio"] == anio) & (puntos["muertes"] > 0)]
    cuadros_muertes.append(del_anio[["latitud", "longitud", "peso_muertes"]].values.tolist())
    print(anio, "→", len(del_anio), "distritos con muertes")

2020 → 1500 distritos con muertes
2021 → 1714 distritos con muertes
2022 → 1209 distritos con muertes
2023 → 383 distritos con muertes


In [ ]:
NARANJAS = {0.2: "#f6c3ab", 0.5: "#eb6834", 0.8: "#a8401a", 1.0: "#6e2609"}

mapa_muertes = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles=MAPA_BASE)

HeatMapWithTime(
    cuadros_muertes,
    index=["2020", "2021", "2022", "2023"],
    gradient=NARANJAS,
    radius=18,
    max_opacity=0.85,
).add_to(mapa_muertes)

leyenda_muertes = LEYENDA.format(titulo="Muertes por COVID-19", colores=", ".join(NARANJAS.values()))
mapa_muertes.get_root().html.add_child(folium.Element(leyenda_muertes))

mapa_muertes

In [27]:
mapa_muertes.save("mapa_covid_muertes.html")
print("Guardado: mapa_covid_muertes.html")

Guardado: mapa_covid_muertes.html


---
## Lo que hicimos

| Paso | Código |
|---|---|
| Descargar un archivo grande por pedazos | `requests.get(url, stream=True)` + `iter_content()` |
| No volver a descargar | `os.path.exists(destino)` |
| Leer solo dos columnas | `pd.read_csv(..., usecols=[...], dtype=str)` |
| Sacar el año de una fecha en texto | `.str[:4]` |
| Mirar antes de filtrar | `.value_counts(dropna=False)` |
| Contar personas por distrito y año | `.groupby(["UBIGEO", "anio"]).size()` |
| Juntar sin perder distritos | `.merge(..., how="outer")` + `.fillna(0)` |
| Un punto por distrito | `distritos.representative_point()` |
| Intensidad entre 0 y 1 | `(casos / maximo) ** 0.5` |
| Un cuadro por año | una lista de listas `[lat, lon, peso]` |
| Mapa de calor con tiempo | `HeatMapWithTime(cuadros, index=..., gradient=...)` |
| Leyenda | `mapa.get_root().html.add_child(folium.Element(html))` |
| Guardar | `mapa.save("mapa.html")` |

### Lo único que hay que recordar

1. **Un mapa de calor pinta puntos, no polígonos.** Cada distrito es un punto
   con una intensidad entre 0 y 1.
2. **Con datos muy desiguales, la escala decide qué se ve.** Sin la raíz
   cuadrada, el mapa del Perú es un punto sobre Lima.
3. **El máximo debe ser el mismo en todos los años**, o los cuadros no se
   pueden comparar.
4. **Un `HeatMapWithTime` por mapa.** Dos en el mismo mapa se desincronizan.

### Límites de estos mapas

- **Son conteos, no tasas.** Lima domina porque tiene más gente, no
  necesariamente porque le fue peor. Para comparar distritos de distinto
  tamaño habría que dividir entre la población.
- **5 % de los casos no tiene distrito** y no aparece en el mapa.
- **Los casos dependen de cuántas pruebas se hicieron.** En 2023 se testeó
  mucho menos que en 2022. Las muertes son un indicador más firme.

---
## Para practicar

1. Cambia `radius` a 10 y a 30. ¿Qué se gana y qué se pierde?
2. Haz el mapa **por mes** en vez de por año: el mes son los caracteres 5 y 6
   de la fecha (`.str[:6]` da `202212`).
3. Quita la raíz cuadrada (`** 0.5`) y mira qué queda del mapa.